In [1]:
import math


# ------------------------------------------------------------------
# Helper: Euclidean distance between two points
# ------------------------------------------------------------------
def distance(p, q):
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(p, q)))


# ------------------------------------------------------------------
# Our sample data (10 points in 2-D)
# ------------------------------------------------------------------
X = [
    (0, 0),  # x1
    (3, 8),  # x2
    (2, 2),  # x3
    (1, 1),  # x4
    (5, 3),  # x5
    (4, 8),  # x6
    (6, 3),  # x7
    (5, 4),  # x8
    (6, 4),  # x9
    (7, 5),  # x10
]


def name(i):
    """Index 0 -> 'x1', index 1 -> 'x2', ..."""
    return f"x{i + 1}"

In [2]:
# ==================================================================
# 1) SIMPLE CLUSTER SEEKING ALGORITHM
# ==================================================================
# Idea: walk through the points ONE BY ONE.
#   - The first point becomes the first cluster centre.
#   - For every next point, find the nearest existing centre.
#       * If that distance <= T   -> join that cluster.
#       * If that distance >  T   -> this point becomes a NEW centre.
# T (threshold) is chosen by you.
# ==================================================================
def simple_cluster_seeking(points, T):
    print(f"\n===== SIMPLE CLUSTER SEEKING  (T = {T}) =====")

    centres = [0]                 # store indexes of centre points
    labels = [0] * len(points)    # cluster number of every point
    print(f"Step 1: {name(0)}{points[0]} becomes centre z1")

    for i in range(1, len(points)):
        # distance from this point to every existing centre
        dists = [distance(points[i], points[c]) for c in centres]
        nearest = dists.index(min(dists))
        d_text = ", ".join(f"z{k + 1}={d:.2f}" for k, d in enumerate(dists))

        if dists[nearest] > T:
            centres.append(i)
            labels[i] = len(centres) - 1
            print(f"{name(i)}{points[i]}: distances [{d_text}] "
                  f"-> all > {T}  => NEW centre z{len(centres)}")
        else:
            labels[i] = nearest
            print(f"{name(i)}{points[i]}: distances [{d_text}] "
                  f"-> join cluster z{nearest + 1}")

    print_result(points, centres, labels)
    return centres, labels



In [4]:
def print_result(points, centres, labels):
    print("\nFINAL CLUSTERS:")
    for k, c in enumerate(centres):
        members = [name(i) for i in range(len(points)) if labels[i] == k]
        print(f"  Cluster {k + 1} (centre {name(c)}{points[c]}): {members}")

In [5]:
if __name__ == "__main__":
    s_centres, s_labels = simple_cluster_seeking(X, T=3)
    
   


===== SIMPLE CLUSTER SEEKING  (T = 3) =====
Step 1: x1(0, 0) becomes centre z1
x2(3, 8): distances [z1=8.54] -> all > 3  => NEW centre z2
x3(2, 2): distances [z1=2.83, z2=6.08] -> join cluster z1
x4(1, 1): distances [z1=1.41, z2=7.28] -> join cluster z1
x5(5, 3): distances [z1=5.83, z2=5.39] -> all > 3  => NEW centre z3
x6(4, 8): distances [z1=8.94, z2=1.00, z3=5.10] -> join cluster z2
x7(6, 3): distances [z1=6.71, z2=5.83, z3=1.00] -> join cluster z3
x8(5, 4): distances [z1=6.40, z2=4.47, z3=1.00] -> join cluster z3
x9(6, 4): distances [z1=7.21, z2=5.00, z3=1.41] -> join cluster z3
x10(7, 5): distances [z1=8.60, z2=5.00, z3=2.83] -> join cluster z3

FINAL CLUSTERS:
  Cluster 1 (centre x1(0, 0)): ['x1', 'x3', 'x4']
  Cluster 2 (centre x2(3, 8)): ['x2', 'x6']
  Cluster 3 (centre x5(5, 3)): ['x5', 'x7', 'x8', 'x9', 'x10']


In [6]:
# ==================================================================
# 2) MAXIMIN (MAX-MIN) DISTANCE ALGORITHM
# ==================================================================
# Idea: pick centres that are as FAR APART as possible.
#   Step 1: first point = centre z1.
#   Step 2: point FARTHEST from z1 = centre z2.
#   Step 3: for every remaining point, find its distance to its
#           NEAREST centre (the "min").
#           Take the point whose min-distance is LARGEST (the "max").
#           -> that is why it is called MAX-MIN.
#   Step 4: if that max-min distance > fraction * (average distance
#           between existing centres), make it a new centre and go
#           back to Step 3. Otherwise STOP.
#   Step 5: assign every point to its nearest centre.
# fraction is usually 1/2.
# ==================================================================
def maximin(points, fraction=0.5):
    print(f"\n===== MAXIMIN DISTANCE ALGORITHM  (fraction = {fraction}) =====")

    # Step 1
    centres = [0]
    print(f"Step 1: {name(0)}{points[0]} becomes centre z1")

    # Step 2: farthest point from z1
    d_from_z1 = [distance(p, points[0]) for p in points]
    far = d_from_z1.index(max(d_from_z1))
    centres.append(far)
    print(f"Step 2: farthest point from z1 is {name(far)}{points[far]} "
          f"(distance {d_from_z1[far]:.2f}) -> centre z2")

    # Steps 3-4: keep adding centres
    while True:
        # average distance between all pairs of existing centres
        pair_d = [distance(points[a], points[b])
                  for idx, a in enumerate(centres) for b in centres[idx + 1:]]
        avg = sum(pair_d) / len(pair_d)
        limit = fraction * avg

        # min distance of each non-centre point to its nearest centre
        best_point, best_d = None, -1
        print(f"\n  Current centres: {[name(c) for c in centres]}")
        for i, p in enumerate(points):
            if i in centres:
                continue
            min_d = min(distance(p, points[c]) for c in centres)
            print(f"    {name(i)}{p}: distance to nearest centre = {min_d:.2f}")
            if min_d > best_d:
                best_point, best_d = i, min_d

        print(f"  MAX of these minimums = {best_d:.2f} at {name(best_point)}")
        print(f"  Limit = {fraction} x average centre distance "
              f"({avg:.2f}) = {limit:.2f}")

        if best_d > limit:
            centres.append(best_point)
            print(f"  {best_d:.2f} > {limit:.2f}  => {name(best_point)} "
                  f"becomes centre z{len(centres)}")
        else:
            print(f"  {best_d:.2f} <= {limit:.2f}  => STOP, no more centres")
            break

    # Step 5: assign each point to its nearest centre
    labels = []
    for p in points:
        d = [distance(p, points[c]) for c in centres]
        labels.append(d.index(min(d)))

    print_result(points, centres, labels)
    return centres, labels

In [7]:
def print_result(points, centres, labels):
    print("\nFINAL CLUSTERS:")
    for k, c in enumerate(centres):
        members = [name(i) for i in range(len(points)) if labels[i] == k]
        print(f"  Cluster {k + 1} (centre {name(c)}{points[c]}): {members}")

In [8]:
if __name__ == "__main__":
    
    m_centres, m_labels = maximin(X, fraction=0.5)


===== MAXIMIN DISTANCE ALGORITHM  (fraction = 0.5) =====
Step 1: x1(0, 0) becomes centre z1
Step 2: farthest point from z1 is x6(4, 8) (distance 8.94) -> centre z2

  Current centres: ['x1', 'x6']
    x2(3, 8): distance to nearest centre = 1.00
    x3(2, 2): distance to nearest centre = 2.83
    x4(1, 1): distance to nearest centre = 1.41
    x5(5, 3): distance to nearest centre = 5.10
    x7(6, 3): distance to nearest centre = 5.39
    x8(5, 4): distance to nearest centre = 4.12
    x9(6, 4): distance to nearest centre = 4.47
    x10(7, 5): distance to nearest centre = 4.24
  MAX of these minimums = 5.39 at x7
  Limit = 0.5 x average centre distance (8.94) = 4.47
  5.39 > 4.47  => x7 becomes centre z3

  Current centres: ['x1', 'x6', 'x7']
    x2(3, 8): distance to nearest centre = 1.00
    x3(2, 2): distance to nearest centre = 2.83
    x4(1, 1): distance to nearest centre = 1.41
    x5(5, 3): distance to nearest centre = 1.00
    x8(5, 4): distance to nearest centre = 1.41
    x9(6